In [2]:

import numpy as np
import pandas as pd
from pathlib import Path
 
DATA_PATH  = Path("../data/raw/creditcard.csv")
OUT_PATH   = Path("../data/processed")
 

In [3]:

df = pd.read_csv(DATA_PATH)
print(f"Loaded: {df.shape[0]:,} rows × {df.shape[1]} cols")

Loaded: 284,807 rows × 31 cols


In [4]:

print("\n--- Feature Engineering ---")
 
# ── 2a. log1p(Amount) ────────────────────────────────────────────────────────
# Amount is right-skewed (median €22, max €25k).
# log1p compresses the tail so a €25k txn isn't 1000× louder than a €25 txn
# in the RBF kernel's distance calculation.
# log1p (not log) because log(0) = -inf and some amounts are €0.00.
df['Amount_log'] = np.log1p(df['Amount'])
 
print(f"Amount_log: mean={df['Amount_log'].mean():.3f}, "
      f"std={df['Amount_log'].std():.3f}  "
      f"(was mean={df['Amount'].mean():.1f}, std={df['Amount'].std():.1f})")
 
# ── 2b. Hour of day (0-23, cyclical) ─────────────────────────────────────────
# Raw hour is 0-47 (two days). We want 0-23 repeating.
# Cyclical encoding: hour 23 and hour 0 should be "close" in feature space,
# but numerically 23 and 0 are far apart.
# Solution: sin/cos encoding maps the cycle onto a unit circle.
#   hour=0  → (sin=0.0,  cos=1.0)
#   hour=6  → (sin=1.0,  cos=0.0)
#   hour=12 → (sin=0.0,  cos=-1.0)
#   hour=23 → (sin=-0.27, cos=0.96) — close to hour=0 ✓
df['Hour'] = (df['Time'] / 3600).astype(int) % 24
df['Hour_sin'] = np.sin(2 * np.pi * df['Hour'] / 24)
df['Hour_cos'] = np.cos(2 * np.pi * df['Hour'] / 24)
 
print(f"Hour_sin range: [{df['Hour_sin'].min():.3f}, "
      f"{df['Hour_sin'].max():.3f}]")
 
# ── 2c. Amount relative to its rolling context ────────────────────────────────
# A €500 txn is suspicious if you usually spend €20.
# We approximate this with a global median ratio since we don't have
# per-customer history in this dataset.
# In a real system you'd compute per-card rolling median.
global_median = df['Amount'].median()
df['Amount_to_median'] = df['Amount'] / (global_median + 1e-9)
 
print(f"Amount_to_median: median={df['Amount_to_median'].median():.3f} "
      f"(should be ≈1.0), fraud mean="
      f"{df[df['Class']==1]['Amount_to_median'].mean():.3f}")
 
# ── 2d. Is this a micro-transaction? ─────────────────────────────────────────
# Fraudsters often make €0.01-€1.00 "ping" transactions to test if a
# stolen card is active before making large purchases.
# EDA showed fraud 25th percentile is very low.
df['Is_micro'] = (df['Amount'] < 1.0).astype(np.float64)
 
fraud_micro_rate = df[df['Class']==1]['Is_micro'].mean()
legit_micro_rate = df[df['Class']==0]['Is_micro'].mean()
print(f"Is_micro: fraud rate={100*fraud_micro_rate:.1f}%, "
      f"legit rate={100*legit_micro_rate:.1f}%")
 
# ── 2e. Time since start (normalized) ────────────────────────────────────────
# Raw Time ranges 0–172792 seconds. We'll scale it but also keep a
# normalized version in [0,1] for interpretability.
df['Time_norm'] = df['Time'] / df['Time'].max()
 
# ── 2f. Drop raw Time and Amount (replaced by engineered versions) ────────────
# Keep original Amount for business reporting, drop from features.
print(f"\nOriginal feature count: {df.shape[1]}")
FEATURES_TO_DROP = ['Time', 'Amount', 'Hour']  # Hour replaced by sin/cos
df_featured = df.drop(columns=FEATURES_TO_DROP)
print(f"After feature engineering: {df_featured.shape[1]} columns")
 


--- Feature Engineering ---
Amount_log: mean=3.152, std=1.657  (was mean=88.3, std=250.1)
Hour_sin range: [-1.000, 1.000]
Amount_to_median: median=1.000 (should be ≈1.0), fraud mean=5.555
Is_micro: fraud rate=13.8%, legit rate=5.9%

Original feature count: 38
After feature engineering: 35 columns


In [5]:

print("\n--- Outlier Decision ---")
 
# Check for extreme V-feature values
v_cols = [f'V{i}' for i in range(1, 29)]
extremes = (np.abs(df[v_cols]) > 10).any(axis=1).sum()
print(f"Rows with any |V| > 10: {extremes:,}")
print(f"Fraud among those rows: "
      f"{df[df[v_cols].abs().gt(10).any(axis=1)]['Class'].mean():.1%}")
print("→ High fraud concentration in extremes. Keep all rows.")
 


--- Outlier Decision ---
Rows with any |V| > 10: 2,316
Fraud among those rows: 9.2%
→ High fraud concentration in extremes. Keep all rows.


In [6]:

print("\n--- Scaling ---")
 
feature_cols = [c for c in df_featured.columns if c != 'Class']
target_col   = 'Class'
 
X = df_featured[feature_cols].values.astype(np.float64)
y = df_featured[target_col].values.astype(np.int32)
 
# CRITICAL: fit scaler only on training data
# We do the train/test split BEFORE fitting the scaler.
# If you scale on all data first, test set statistics leak into training.
 
# Stratified split — preserves class ratio in both sets
rng = np.random.RandomState(42)
fraud_idx   = np.where(y == 1)[0]
legit_idx   = np.where(y == 0)[0]
 
# 80/20 split, stratified
n_fraud_test = int(len(fraud_idx) * 0.20)
n_legit_test = int(len(legit_idx) * 0.20)
 
rng.shuffle(fraud_idx)
rng.shuffle(legit_idx)
 
test_idx  = np.concatenate([fraud_idx[:n_fraud_test],
                             legit_idx[:n_legit_test]])
train_idx = np.concatenate([fraud_idx[n_fraud_test:],
                             legit_idx[n_legit_test:]])
 
X_train, X_test = X[train_idx], X[test_idx]
y_train, y_test = y[train_idx], y[test_idx]
 
print(f"Train: {len(X_train):,} samples  "
      f"(fraud: {y_train.sum()}, legit: {(y_train==0).sum()})")
print(f"Test:  {len(X_test):,} samples   "
      f"(fraud: {y_test.sum()}, legit: {(y_test==0).sum()})")
 
# Fit scaler on train only
train_mean = X_train.mean(axis=0)
train_std  = X_train.std(axis=0)
train_std[train_std == 0] = 1.0  # Avoid divide-by-zero for constant features
 
X_train_scaled = (X_train - train_mean) / train_std
X_test_scaled  = (X_test  - train_mean) / train_std  # ← train stats, not test
 
print(f"\nPost-scaling train stats (should be ≈ mean=0, std=1):")
print(f"  mean: min={X_train_scaled.mean(axis=0).min():.4f}, "
      f"max={X_train_scaled.mean(axis=0).max():.4f}")
print(f"  std:  min={X_train_scaled.std(axis=0).min():.4f}, "
      f"max={X_train_scaled.std(axis=0).max():.4f}")
 


--- Scaling ---
Train: 227,846 samples  (fraud: 394, legit: 227452)
Test:  56,961 samples   (fraud: 98, legit: 56863)

Post-scaling train stats (should be ≈ mean=0, std=1):
  mean: min=-0.0000, max=0.0000
  std:  min=1.0000, max=1.0000


In [7]:

print("\n--- Development Subsample ---")
 
fraud_train_idx = np.where(y_train == 1)[0]
legit_train_idx = np.where(y_train == 0)[0]
 
N_LEGIT_SAMPLE = 4607  # → total ~5000
 
rng.shuffle(legit_train_idx)
dev_idx = np.concatenate([
    fraud_train_idx,                          # all fraud
    legit_train_idx[:N_LEGIT_SAMPLE]          # sampled legit
])
rng.shuffle(dev_idx)
 
X_dev = X_train_scaled[dev_idx]
y_dev = y_train[dev_idx]
 
print(f"Dev set: {len(X_dev):,} samples")
print(f"  Fraud: {y_dev.sum()} ({100*y_dev.mean():.2f}%)")
print(f"  Legit: {(y_dev==0).sum()}")
print(f"  Imbalance ratio: {(y_dev==0).sum()/y_dev.sum():.1f}:1  "
      f"(was 578:1, now {(y_dev==0).sum()/y_dev.sum():.1f}:1)")
print(f"  Kernel matrix size: {len(X_dev)**2 * 8 / 1e6:.1f} MB  ✓")
 


--- Development Subsample ---
Dev set: 5,001 samples
  Fraud: 394 (7.88%)
  Legit: 4607
  Imbalance ratio: 11.7:1  (was 578:1, now 11.7:1)
  Kernel matrix size: 200.1 MB  ✓


In [8]:
 
print("\n--- Class Weights ---")
 
n_train  = len(y_train)
n_fraud  = y_train.sum()
n_legit  = (y_train == 0).sum()
n_classes = 2
 
# sklearn's 'balanced' formula:
weight_legit = n_train / (n_classes * n_legit)
weight_fraud = n_train / (n_classes * n_fraud)
 
print(f"Balanced class weights (for full training set):")
print(f"  Legit weight: {weight_legit:.4f}")
print(f"  Fraud weight: {weight_fraud:.4f}")
print(f"  Ratio: {weight_fraud/weight_legit:.1f}  "
      f"(fraud penalized {weight_fraud/weight_legit:.1f}× harder)")
 
# For our dev set (different ratio):
n_dev_fraud = y_dev.sum()
n_dev_legit = (y_dev == 0).sum()
weight_fraud_dev = len(y_dev) / (2 * n_dev_fraud)
weight_legit_dev = len(y_dev) / (2 * n_dev_legit)
 
print(f"\nBalanced class weights (for dev subsample):")
print(f"  Legit weight: {weight_legit_dev:.4f}")
print(f"  Fraud weight: {weight_fraud_dev:.4f}")
print(f"  Ratio: {weight_fraud_dev/weight_legit_dev:.1f}")
 
print("\n  How class weights enter the SVM loss:")
print("  Instead of: min  (1/2)||w||² + C Σ ξᵢ")
print("  We solve:   min  (1/2)||w||² + Σ Cᵢ ξᵢ")
print("  Where Cᵢ = C × weight[yᵢ]")
print("  → Misclassifying a fraud costs 11.7× more than misclassifying legit")
 


--- Class Weights ---
Balanced class weights (for full training set):
  Legit weight: 0.5009
  Fraud weight: 289.1447
  Ratio: 577.3  (fraud penalized 577.3× harder)

Balanced class weights (for dev subsample):
  Legit weight: 0.5428
  Fraud weight: 6.3464
  Ratio: 11.7

  How class weights enter the SVM loss:
  Instead of: min  (1/2)||w||² + C Σ ξᵢ
  We solve:   min  (1/2)||w||² + Σ Cᵢ ξᵢ
  Where Cᵢ = C × weight[yᵢ]
  → Misclassifying a fraud costs 11.7× more than misclassifying legit


In [ ]:

print("\n--- Evaluation Functions ---")
 
def evaluate(y_true, y_pred, y_score=None, threshold=0.5, label="Model"):
    """
    Comprehensive evaluation for imbalanced binary classification.
    
    y_true  : true labels {0, 1}
    y_pred  : predicted labels {0, 1}
    y_score : raw decision scores (for AUC calculation)
    """
    # Confusion matrix elements
    TP = np.sum((y_pred == 1) & (y_true == 1))
    TN = np.sum((y_pred == 0) & (y_true == 0))
    FP = np.sum((y_pred == 1) & (y_true == 0))
    FN = np.sum((y_pred == 0) & (y_true == 1))
 
    precision  = TP / (TP + FP + 1e-10)
    recall     = TP / (TP + FN + 1e-10)   # = sensitivity = TPR
    f1         = 2 * precision * recall / (precision + recall + 1e-10)
    accuracy   = (TP + TN) / len(y_true)
    specificity = TN / (TN + FP + 1e-10)  # TNR
 
    print(f"\n{'='*50}")
    print(f"  {label}")
    print(f"{'='*50}")
    print(f"  Confusion Matrix:")
    print(f"              Pred Legit   Pred Fraud")
    print(f"  True Legit  {TN:>10,}   {FP:>10,}")
    print(f"  True Fraud  {FN:>10,}   {TP:>10,}")
    print(f"\n  Accuracy:    {100*accuracy:.3f}%  ← IGNORE THIS")
    print(f"  Precision:   {100*precision:.2f}%  "
          f"(of flagged txns, how many are real fraud)")
    print(f"  Recall:      {100*recall:.2f}%  "
          f"(of real frauds, how many did we catch)")
    print(f"  Specificity: {100*specificity:.3f}%  "
          f"(of legit txns, how many passed through)")
    print(f"  F1 Score:    {f1:.4f}")
    print(f"  False Positives: {FP:,}  "
          f"(legit txns incorrectly flagged)")
    print(f"  False Negatives: {FN:,}  (frauds we missed)")
 
    if y_score is not None:
        auc_pr = compute_auc_pr(y_true, y_score)
        auc_roc = compute_auc_roc(y_true, y_score)
        print(f"  AUC-PR:      {auc_pr:.4f}  ← PRIMARY METRIC")
        print(f"  AUC-ROC:     {auc_roc:.4f}  "
              f"(optimistic on imbalanced data, use cautiously)")
 
    # Business impact
    # Assume: avg fraud amount €122 (from EDA), false positive review cost €2
    avg_fraud_loss   = 122
    fp_review_cost   = 2
    caught_value     = TP * avg_fraud_loss
    missed_loss      = FN * avg_fraud_loss
    fp_cost          = FP * fp_review_cost
    net_value        = caught_value - missed_loss - fp_cost
 
    print(f"\n  Business Impact (estimated):")
    print(f"  Fraud caught:        €{caught_value:>10,.0f}")
    print(f"  Fraud missed:       -€{missed_loss:>10,.0f}")
    print(f"  FP review cost:     -€{fp_cost:>10,.0f}")
    print(f"  Net value:           €{net_value:>10,.0f}")
 
    return {
        'precision': precision, 'recall': recall, 'f1': f1,
        'accuracy': accuracy, 'TP': TP, 'TN': TN, 'FP': FP, 'FN': FN,
    }
 
 
def compute_auc_pr(y_true, y_score):
    """
    Area under Precision-Recall curve via trapezoidal integration.
    
    We vary the threshold from high to low, computing precision and recall
    at each point. The area summarizes performance across ALL thresholds.
    
    Why AUC-PR instead of AUC-ROC for imbalanced data?
    
    AUC-ROC plots TPR vs FPR. With 284k legit transactions, even a 0.1% FPR
    means 284 false positives — that looks great on the ROC curve but is
    terrible in practice. AUC-PR focuses on the minority class directly.
    
    A random classifier gets AUC-PR ≈ fraud_rate (0.17% here).
    Our model should be >> 0.17%.
    """
    # Sort by descending score
    sorted_idx = np.argsort(y_score)[::-1]
    y_true_sorted = y_true[sorted_idx]
 
    # Cumulative TP and FP as we lower threshold
    cum_tp = np.cumsum(y_true_sorted)
    cum_fp = np.cumsum(1 - y_true_sorted)
    n_pos  = y_true.sum()
 
    precision_curve = cum_tp / (cum_tp + cum_fp + 1e-10)
    recall_curve    = cum_tp / (n_pos + 1e-10)
 
    # Prepend (recall=0, precision=1) for proper curve start
    precision_curve = np.concatenate([[1.0], precision_curve])
    recall_curve    = np.concatenate([[0.0], recall_curve])
 
    # Trapezoidal integration: AUC = ∫ precision d(recall)
    auc = np.trapz(precision_curve, recall_curve)
    return abs(auc)
 
 
def compute_auc_roc(y_true, y_score):
    """
    AUC-ROC via Mann-Whitney U statistic.
    
    AUC = P(score(positive) > score(negative))
        = fraction of (fraud, legit) pairs where fraud 
          is scored higher than legit
    
    Equivalent to the trapezoidal ROC curve but numerically stable.
    Uses rank-sum formula: O(n log n) via sorting.
    """
    n_pos = int(y_true.sum())
    n_neg = len(y_true) - n_pos
    
    if n_pos == 0 or n_neg == 0:
        return 0.5
    
    # Rank all scores from 1 (lowest) to n (highest)
    # np.argsort twice gives ranks
    ranks = np.argsort(np.argsort(y_score)) + 1
    
    # Sum of ranks belonging to the positive class
    rank_sum_pos = np.sum(ranks[y_true == 1])
    
    # U statistic for positives
    U = rank_sum_pos - n_pos * (n_pos + 1) / 2
    
    # Normalize to [0, 1]
    return float(U / (n_pos * n_neg))
 
 
def find_threshold_for_recall(y_true, y_score, target_recall=0.90):
    """
    Find the decision threshold that achieves a target recall.
    
    Business use case: "We want to catch at least 90% of frauds.
    What threshold should we set, and what's the precision at that point?"
    """
    thresholds = np.sort(y_score)[::-1]
    best_thresh = 0.5
    best_precision = 0.0
 
    for t in thresholds:
        y_pred_t = (y_score >= t).astype(int)
        TP = np.sum((y_pred_t == 1) & (y_true == 1))
        FP = np.sum((y_pred_t == 1) & (y_true == 0))
        FN = np.sum((y_pred_t == 0) & (y_true == 1))
 
        recall_t    = TP / (TP + FN + 1e-10)
        precision_t = TP / (TP + FP + 1e-10)
 
        if recall_t >= target_recall:
            best_thresh    = t
            best_precision = precision_t
            break
 
    return best_thresh, best_precision
 


--- Evaluation Functions ---


In [10]:

print("\n--- Saving Processed Data ---")
 
np.save(OUT_PATH / "X_train_scaled.npy",  X_train_scaled)
np.save(OUT_PATH / "X_test_scaled.npy",   X_test_scaled)
np.save(OUT_PATH / "y_train.npy",         y_train)
np.save(OUT_PATH / "y_test.npy",          y_test)
np.save(OUT_PATH / "X_dev.npy",           X_dev)
np.save(OUT_PATH / "y_dev.npy",           y_dev)
 
# Save scaler parameters (so we can scale new transactions at serve time)
np.save(OUT_PATH / "scaler_mean.npy",     train_mean)
np.save(OUT_PATH / "scaler_std.npy",      train_std)
 
# Save feature names (needed for the serving layer)
import json
meta = {
    "feature_names":    feature_cols,
    "n_features":       len(feature_cols),
    "n_train":          int(len(X_train)),
    "n_test":           int(len(X_test)),
    "n_dev":            int(len(X_dev)),
    "fraud_rate_train": float(y_train.mean()),
    "fraud_rate_dev":   float(y_dev.mean()),
    "class_weight_fraud_full": float(weight_fraud),
    "class_weight_fraud_dev":  float(weight_fraud_dev),
    "class_weight_legit_full": float(weight_legit),
    "class_weight_legit_dev":  float(weight_legit_dev),
}
with open(OUT_PATH / "dataset_meta.json", "w") as f:
    json.dump(meta, f, indent=2)
 
print(f"Saved to {OUT_PATH}/:")
for p in sorted(OUT_PATH.iterdir()):
    size = p.stat().st_size / 1e6
    print(f"  {p.name:<35} {size:.1f} MB")


--- Saving Processed Data ---
Saved to ../data/processed/:
  X_dev.npy                           1.4 MB
  X_test_scaled.npy                   15.5 MB
  X_train_scaled.npy                  62.0 MB
  dataset_meta.json                   0.0 MB
  scaler_mean.npy                     0.0 MB
  scaler_std.npy                      0.0 MB
  y_dev.npy                           0.0 MB
  y_test.npy                          0.2 MB
  y_train.npy                         0.9 MB


In [11]:

print("\n--- Baseline: Predict Everything Legit ---")
 
y_baseline = np.zeros(len(y_test), dtype=int)
evaluate(y_test, y_baseline, label="Always-Legit Baseline")
 
print("\n--- Baseline: Predict Everything Fraud ---")
 
y_all_fraud = np.ones(len(y_test), dtype=int)
evaluate(y_test, y_all_fraud, label="Always-Fraud Baseline")
 
print("""
Summary:
  Always-Legit: 99.8% accuracy, 0% recall. Useless.
  Always-Fraud: 0.17% precision, 100% recall. Flags everything.
  
  Our SVM must operate somewhere in between, tuned by the business
  to hit their acceptable tradeoff of false positives vs false negatives.
  
  The AUC-PR score for a random classifier on this dataset ≈ 0.0017.
  A good model should exceed 0.70. Elite models reach 0.85+.
""")
 
print("=" * 60)
print("PREPROCESSING COMPLETE")
print(f"  Dev set ready:   {X_dev.shape}  "
      f"({100*y_dev.mean():.1f}% fraud)")
print(f"  Test set ready:  {X_test_scaled.shape}  "
      f"({100*y_test.mean():.1f}% fraud)")
print(f"  Features:        {len(feature_cols)}")
print("=" * 60)
 


--- Baseline: Predict Everything Legit ---

  Always-Legit Baseline
  Confusion Matrix:
              Pred Legit   Pred Fraud
  True Legit      56,863            0
  True Fraud          98            0

  Accuracy:    99.828%  ← IGNORE THIS
  Precision:   0.00%  (of flagged txns, how many are real fraud)
  Recall:      0.00%  (of real frauds, how many did we catch)
  Specificity: 100.000%  (of legit txns, how many passed through)
  F1 Score:    0.0000
  False Positives: 0  (legit txns incorrectly flagged)
  False Negatives: 98  (frauds we missed)

  Business Impact (estimated):
  Fraud caught:        €         0
  Fraud missed:       -€    11,956
  FP review cost:     -€         0
  Net value:           €   -11,956

--- Baseline: Predict Everything Fraud ---

  Always-Fraud Baseline
  Confusion Matrix:
              Pred Legit   Pred Fraud
  True Legit           0       56,863
  True Fraud           0           98

  Accuracy:    0.172%  ← IGNORE THIS
  Precision:   0.17%  (of flagged